# 03 — ML-Based Fraud Detection with LightGBM

**Goal:** Train a LightGBM binary classifier on rule engine outputs + engineered features
to improve SAR precision — mirroring the hybrid rule+ML approach used by modern AML platforms.

| Approach | Precision | Recall | Scalability |
|---|---|---|---|
| Rules only | Low–Medium | High | High |
| ML only | Medium | Medium | High |
| **Rules + ML (this notebook)** | **High** | **High** | **High** |

In [ ]:
import os

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
for param in ['axes.facecolor', 'figure.facecolor', 'savefig.facecolor']:
    plt.rcParams[param] = '#0a0d16'
plt.rcParams['axes.edgecolor'] = '#181e30'

ACCENT = '#00c2f0'; WARN = '#f5a623'; DANGER = '#e05a5a'; GREEN = '#22d46a'
print(f'✓ LightGBM {lgb.__version__} | Working dir: {os.getcwd()}')

## Step 1 — Load Scored Data

In [ ]:
if not os.path.exists('outputs/scored_transactions.csv'):
    %run generate_data.py
    from detect_anomalies import compute_risk_score, save_flagged
    raw = pd.read_csv('outputs/raw_transactions.csv', parse_dates=['timestamp'])
    scored = compute_risk_score(raw)
    scored.to_csv('outputs/scored_transactions.csv', index=False)
else:
    scored = pd.read_csv('outputs/scored_transactions.csv', parse_dates=['timestamp'])

print(f'Shape: {scored.shape}')
scored.head(3)

## Step 2 — Feature Engineering

In [ ]:
df = scored.copy()

# Time features
df['hour']              = df['timestamp'].dt.hour
df['day_of_week']       = df['timestamp'].dt.dayofweek
df['is_weekend']        = (df['day_of_week'] >= 5).astype(int)
df['is_late_night']     = df['hour'].between(1, 3).astype(int)
df['is_business_hours'] = df['hour'].between(8, 18).astype(int)

# Amount features
df['amount_log'] = np.log1p(df['amount_bdt'])
sender_stats = df.groupby('sender_id')['amount_bdt'].agg(
    sender_median='median', sender_mean='mean', sender_txn_count='count'
).reset_index()
df = df.merge(sender_stats, on='sender_id', how='left')
df['amount_vs_median_ratio'] = df['amount_bdt'] / (df['sender_median'] + 1)
df['amount_vs_mean_ratio']   = df['amount_bdt'] / (df['sender_mean'] + 1)
df['is_outlier_amount']      = (df['amount_vs_median_ratio'] > 5).astype(int)

# Encode tx_type
te = LabelEncoder()
df['tx_type_enc'] = te.fit_transform(df['tx_type'])

# Target
df['is_anomaly'] = (df['anomaly_flag'] != 'NORMAL').astype(int)

print(f'Class distribution: {df["is_anomaly"].value_counts().to_dict()}')
print(f'Positive rate: {df["is_anomaly"].mean()*100:.2f}%')

In [ ]:
RULE_COLS = [c for c in df.columns if c.startswith('RULE_')]
FEATURE_COLS = (
    RULE_COLS + ['risk_score'] +
    ['hour','day_of_week','is_weekend','is_late_night','is_business_hours'] +
    ['amount_log','amount_vs_median_ratio','amount_vs_mean_ratio','is_outlier_amount','sender_txn_count'] +
    ['tx_type_enc']
)
print(f'Total features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

## Step 3 — Class Imbalance

In [ ]:
n_pos = df['is_anomaly'].sum()
n_neg = (df['is_anomaly'] == 0).sum()
ratio = n_neg / n_pos
print(f'Normal (0):  {n_neg:,}')
print(f'Anomaly (1): {n_pos:,}')
print(f'Imbalance ratio: {ratio:.1f}:1')
print(f'\nStrategy: scale_pos_weight = {ratio:.2f}')
print('Why NOT SMOTE: RULE_* features are binary — interpolation creates impossible values.')

## Step 4 — Train / Test Split

In [ ]:
X = df[FEATURE_COLS].fillna(0)
y = df['is_anomaly']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  ({y_train.sum()} anomalies)')
print(f'Test:  {len(X_test):,}  ({y_test.sum()} anomalies)')

## Step 5 — LightGBM Training

**Why LightGBM?** Handles class imbalance natively, leaf-wise growth beats sklearn GBM on tabular financial data, scales to millions of rows, feature importance for compliance explainability.

In [ ]:
scale_pos = (y_train == 0).sum() / y_train.sum()

model = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31,
    min_child_samples=10, scale_pos_weight=scale_pos,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, verbose=-1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)]
)
print(f'\nBest iteration: {model.best_iteration_} ✓')

## Step 6 — Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
auc  = roc_auc_score(y_test, y_prob)
avgp = average_precision_score(y_test, y_prob)

print('=== LightGBM ===')
print(classification_report(y_test, y_pred, target_names=['NORMAL','ANOMALY'], digits=4))
print(f'ROC-AUC:    {auc:.4f}')
print(f'PR-AUC:     {avgp:.4f}')

print('\n=== Rule Engine (score >= 30) ===')
rule_pred = (X_test['risk_score'] >= 30).astype(int)
print(classification_report(y_test, rule_pred, target_names=['NORMAL','ANOMALY'], digits=4))
print(f'F1 improvement: +{f1_score(y_test, y_pred) - f1_score(y_test, rule_pred):.4f}')

## Step 7 — Confusion Matrix & Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cm = confusion_matrix(y_test, y_pred)
axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(['NORMAL','ANOMALY'])
axes[0].set_yticks([0,1]); axes[0].set_yticklabels(['NORMAL','ANOMALY'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix', color='white', pad=12)
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                    color='white' if cm[i,j] < cm.max()/2 else 'black', fontsize=12)

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color=ACCENT, linewidth=2, label=f'LightGBM (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1], color='gray', linestyle='--', linewidth=1)
axes[1].set_title('ROC Curve', color='white', pad=12)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend(fontsize=9)

prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(rec, prec, color=GREEN, linewidth=2, label=f'AP={avgp:.3f}')
axes[2].set_title('Precision-Recall Curve', color='white', pad=12)
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].legend(fontsize=9); axes[2].set_xlim([0,1]); axes[2].set_ylim([0,1.05])

plt.tight_layout()
os.makedirs('images', exist_ok=True)
plt.savefig('images/03_eval_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 8 — Feature Importance

> Regulators require SAR decisions to be justifiable. LightGBM feature importance provides this transparency.

In [ ]:
from matplotlib.patches import Patch

importances = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [DANGER if (f.startswith('RULE') or f == 'risk_score') else ACCENT for f in importances['feature']]
ax.barh(importances['feature'], importances['importance'], color=colors, edgecolor='#181e30')
ax.set_title('Feature Importance (red=rule engine, blue=engineered)', color='white', fontsize=12, pad=12)
ax.set_xlabel('Importance (split count)')
ax.legend(handles=[Patch(facecolor=DANGER, label='Rule engine'), Patch(facecolor=ACCENT, label='Engineered')], fontsize=9)
plt.tight_layout()
plt.savefig('images/03_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top 5:', importances.tail(5)[['feature','importance']].to_string(index=False))

## Step 9 — Threshold Tuning

In AML, missing a true SAR (FN) costs far more than a false alert (FP). Tune threshold for recall.

In [ ]:
print(f"{'Threshold':>12} {'Precision':>12} {'Recall':>10} {'F1':>8} {'Flagged':>10}")
print('-' * 58)
best_f1, best_thresh = 0, 0.5
for thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    preds = (y_prob >= thresh).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    n = preds.sum()
    marker = ' ← best F1' if f > best_f1 else ''
    if f > best_f1: best_f1, best_thresh = f, thresh
    print(f'{thresh:>12.1f} {p:>12.3f} {r:>10.3f} {f:>8.3f} {n:>10,}{marker}')
print(f'\nRecommended: {best_thresh} | For max recall (AML): try 0.3')

## Step 10 — Export SAR Candidates

In [ ]:
X_full = df[FEATURE_COLS].fillna(0)
df['ml_score']      = model.predict_proba(X_full)[:, 1]
df['ml_prediction'] = (df['ml_score'] >= best_thresh).astype(int)

sar = df[(df['risk_tier'] == 'HIGH') | (df['ml_score'] >= 0.7)].sort_values('ml_score', ascending=False)
print(f'SAR candidates: {len(sar):,}')
print(f'True anomalies: {sar["is_anomaly"].sum():,} ({sar["is_anomaly"].mean()*100:.1f}%)')

sar[['transaction_id','timestamp','sender_id','receiver_id','amount_bdt','tx_type',
     'division','risk_score','risk_tier','rules_triggered','ml_score','anomaly_flag'
]].to_csv('outputs/sar_candidates.csv', index=False)
print('\nSaved → outputs/sar_candidates.csv')
sar.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df[df['is_anomaly']==0]['ml_score'], bins=50, alpha=0.6, color=GREEN, label='Normal', density=True)
ax.hist(df[df['is_anomaly']==1]['ml_score'], bins=50, alpha=0.7, color=DANGER, label='Anomaly', density=True)
ax.axvline(best_thresh, color=WARN, linestyle='--', linewidth=1.5, label=f'Threshold ({best_thresh})')
ax.set_title('ML Score Distribution by True Label', color='white', fontsize=13, pad=12)
ax.set_xlabel('Predicted Anomaly Probability'); ax.set_ylabel('Density'); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('images/03_score_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## Final Summary

| Output | File |
|---|---|
| Raw transactions | `outputs/raw_transactions.csv` |
| Rule-scored | `outputs/scored_transactions.csv` |
| Rule-flagged | `outputs/flagged_transactions.csv` |
| SAR candidates | `outputs/sar_candidates.csv` |

**5 Key Learnings:**
1. Hybrid rule+ML beats either alone
2. BD-specific calibration is the moat — ROUND_AMOUNT uses sender profile not just BDT value
3. `scale_pos_weight` is the single most important LightGBM param for AML
4. Threshold 0.5 is wrong for imbalanced AML data — always tune
5. Feature importance = compliance explainability — not optional

---
*BD BFIU context · bKash/Nagad patterns · Reproducible synthetic data*